# Coding practice: backprop through a tiny network

Autograd hides the chain rule. Here you turn the crank by hand, verify the result two independent ways, and probe an inactive-ReLU counterexample. The course check uses a changed randomized case, so carry over the method rather than a printed literal.

> Save your own copy first (File -> Save a copy in Drive).

In [ ]:
# One example through h = ReLU(w1*x + b1); y = w2*h + b2; loss = 0.5*(y - target)^2
x, w1, b1, w2, b2, target = 2.0, 0.5, 0.0, -1.0, 1.0, 1.0
z1 = w1 * x + b1          # pre-activation = 1.0
h  = max(0.0, z1)         # ReLU        = 1.0
y  = w2 * h + b2          # output      = 0.0
loss = 0.5 * (y - target) ** 2
print("forward:", dict(z1=z1, h=h, y=y, loss=loss))

In [ ]:
def grad_w1(x, z1, h, y, w2, target):
    """Return dL/dw1 by the chain rule, back to front."""
    dy  = y - target                       # dL/dy
    dh  = dy * w2                           # dL/dh  (through the output layer)
    dz1 = dh * (1.0 if z1 > 0 else 0.0)     # dL/dz1 (through the ReLU)
    # TODO: one more step back through z1 = w1*x + b1, so dL/dw1 = dz1 * x. Return it.
    raise NotImplementedError("Return dL/dw1 = dz1 * x.")

In [ ]:
analytic = grad_w1(x, z1, h, y, w2, target)
print("analytic dL/dw1 =", analytic)

In [ ]:
# Independent checks: centered finite difference and PyTorch autograd.
def loss_at(candidate_w1):
    hidden = max(0.0, candidate_w1 * x + b1)
    prediction = w2 * hidden + b2
    return 0.5 * (prediction - target) ** 2

epsilon = 1e-5
finite_difference = (loss_at(w1 + epsilon) - loss_at(w1 - epsilon)) / (2 * epsilon)

try:
    import torch
    w1_tensor = torch.tensor(w1, requires_grad=True)
    prediction = w2 * torch.relu(w1_tensor * x + b1) + b2
    torch_loss = 0.5 * (prediction - target) ** 2
    torch_loss.backward()
    autograd_value = w1_tensor.grad.item()
except ModuleNotFoundError:
    autograd_value = None
    print("PyTorch is unavailable here; Colab includes it for the autograd cross-check.")

assert abs(analytic - 2.0) < 1e-12
assert abs(analytic - finite_difference) < 1e-6
if autograd_value is not None:
    assert abs(analytic - autograd_value) < 1e-6
print({"analytic": analytic, "finite_difference": finite_difference, "autograd": autograd_value})

In [ ]:
# Changed case: a strictly negative pre-activation blocks this path.
inactive_z1 = -1.0
inactive_h = 0.0
inactive_y = b2
inactive_gradient = grad_w1(x, inactive_z1, inactive_h, inactive_y, w2, target)
assert inactive_gradient == 0.0
print("inactive-ReLU dL/dw1 =", inactive_gradient)

## Interpret the evidence

In your copy, explain why agreement among the analytic trace, finite difference, and autograd is stronger debugging evidence than any one calculation alone. Then explain why the inactive-ReLU case is a counterexample to the claim that a nonzero upstream derivative always gives every earlier weight a nonzero gradient.

Use the same dependency-path method on the randomized course check.